# Aiscern — Video Deepfake Detector Fine-tuning

**Strategy**: Sample frames from videos → fine-tune ViT on frames → aggregate predictions

**Model**: `google/vit-base-patch16-224` (same as image) — no video-specific GPU RAM

**Run on Kaggle** | **Output**: `saghi776/aiscern-video-detector`

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate Pillow huggingface_hub av

In [ ]:
HF_TOKEN     = 'YOUR_HF_TOKEN_HERE'
BASE_MODEL   = 'google/vit-base-patch16-224'
PUSH_TO_REPO = 'saghi776/aiscern-video-detector'
FRAMES_PER_VIDEO = 8    # extract 8 frames per video clip
MAX_SAMPLES  = 4000     # per class (video = fewer samples)
BATCH_SIZE   = 32
EPOCHS       = 5
LR           = 2e-4
import os; os.environ['HF_TOKEN'] = HF_TOKEN
print('Config ✅')

In [ ]:
# Best approach for zero-budget video: use FACE CROP datasets from video deepfakes
# These are already extracted frames — no video decoding needed!
from datasets import load_dataset, concatenate_datasets

print('Loading Celeb-DF deepfake face crops...')
celeb = load_dataset('haywhy/celeb-df-v2', split='train', token=HF_TOKEN)
label_map = {'fake': 1, 'real': 0, 'FAKE': 1, 'REAL': 0, '1': 1, '0': 0, 1: 1, 0: 0}

print('Loading FaceForensics++ face crops...')
ff   = load_dataset('marcelomoreno26/deepfake-detection', split='train', token=HF_TOKEN)

print('Loading DeepFake vs Real faces...')
dvr  = load_dataset('arnabdhar/DeepFake-Vs-Real-Faces', split='train', token=HF_TOKEN)

all_ds = []
for ds in [celeb, ff, dvr]:
    img_col = next((c for c in ['image','img','Image'] if c in ds.column_names), None)
    lbl_col = next((c for c in ['label','Label','is_fake'] if c in ds.column_names), None)
    if img_col and lbl_col:
        ds = ds.map(lambda x: {'label': label_map.get(x[lbl_col], -1)})
        ds = ds.filter(lambda x: x['label'] != -1)
        if img_col != 'image': ds = ds.rename_column(img_col, 'image')
        all_ds.append(ds.select_columns(['image','label']))

combined = concatenate_datasets(all_ds)
real = combined.filter(lambda x: x['label']==0).shuffle(42).select(range(min(MAX_SAMPLES, combined.filter(lambda x: x['label']==0).num_rows)))
fake = combined.filter(lambda x: x['label']==1).shuffle(42).select(range(min(MAX_SAMPLES, combined.filter(lambda x: x['label']==1).num_rows)))
n = min(len(real), len(fake))
balanced = concatenate_datasets([real.select(range(n)), fake.select(range(n))]).shuffle(42)
split = balanced.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'Train: {len(train_ds):,}  Eval: {len(eval_ds):,}')

In [ ]:
# Preprocess + train — identical to image notebook
from transformers import ViTForImageClassification, ViTImageProcessor, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
import evaluate, numpy as np, torch

processor = ViTImageProcessor.from_pretrained(BASE_MODEL)

def preprocess(batch):
    from PIL import Image as PILImage
    imgs = []
    for img in batch['image']:
        if hasattr(img, 'convert'): imgs.append(img.convert('RGB'))
        else: imgs.append(PILImage.fromarray(np.uint8(img)).convert('RGB'))
    batch['pixel_values'] = processor(images=imgs, return_tensors='np')['pixel_values']
    return batch

train_ds = train_ds.map(preprocess, batched=True, batch_size=32, remove_columns=['image'])
eval_ds  = eval_ds.map(preprocess,  batched=True, batch_size=32, remove_columns=['image'])
train_ds.set_format('torch'); eval_ds.set_format('torch')

model = ViTForImageClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0:'real',1:'fake'}, label2id={'real':0,'fake':1},
    ignore_mismatched_sizes=True
)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.1,
    bias='none', target_modules=['query','value','key','dense']))
model.print_trainable_parameters()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

acc = evaluate.load('accuracy'); f1 = evaluate.load('f1')
def compute_metrics(ep):
    preds = np.argmax(ep.predictions, axis=-1)
    return {'accuracy': acc.compute(predictions=preds, references=ep.label_ids)['accuracy'],
            'f1': f1.compute(predictions=preds, references=ep.label_ids, average='binary')['f1']}

args = TrainingArguments(
    output_dir='./video-detector-checkpoints',
    num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE, learning_rate=LR,
    warmup_ratio=0.1, weight_decay=0.01, evaluation_strategy='epoch',
    save_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='f1',
    push_to_hub=True, hub_model_id=PUSH_TO_REPO, hub_token=HF_TOKEN,
    fp16=torch.cuda.is_available(), report_to='none', logging_steps=50,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=eval_ds, compute_metrics=compute_metrics)
trainer.train()
results = trainer.evaluate()
print(f"Accuracy: {results['eval_accuracy']:.4f}  F1: {results['eval_f1']:.4f}")
trainer.push_to_hub(commit_message='Fine-tuned ViT video deepfake detector (face-crop)')
processor.push_to_hub(PUSH_TO_REPO, token=HF_TOKEN)
print(f'✅ https://huggingface.co/{PUSH_TO_REPO}')